In [13]:
import pandas as pd

In [14]:
df = pd.read_csv('/home/gianluca/Research/datapruebas_analysis/data/raw/tmt/neuropruebas/subjects/marialaraa@gmail.com-tmt-plugin_125.csv')

In [15]:
df[df['trial_type'] == 'survey-html-form']['response']

2     {"age":"30","MouseOrPad-choice":"pad de notebo...
50                 {"PadUsechoice":"Con una sola mano"}
Name: response, dtype: object

In [16]:
df[df['trial_type'] == 'survey-html-form']['response'].values

array(['{"age":"30","MouseOrPad-choice":"pad de notebook","hand-choice":"izquierda","hand_config-choice":"no","mail":"marialaraa@gmail.com"}',
       '{"PadUsechoice":"Con una sola mano"}'], dtype=object)

In [17]:
for i in df[df['trial_type'] == 'survey-html-form']['response'].values:
    x = eval(i)



In [18]:
def _read_neuropruebas_survey(df):
    survey_rows = df[df['trial_type'] == 'survey-html-form']['response']
    if len(survey_rows) == 0:
        return None
    survey_response = {}
    for row in survey_rows:
        data = eval(row)
        survey_response.update(data)

    return survey_response

In [19]:
_read_neuropruebas_survey(df)

{'age': '30',
 'MouseOrPad-choice': 'pad de notebook',
 'hand-choice': 'izquierda',
 'hand_config-choice': 'no',
 'mail': 'marialaraa@gmail.com',
 'PadUsechoice': 'Con una sola mano'}

In [20]:
import re

def limpiar_comentario_final(texto: str) -> str:
    """
    Elimina el campo ,"comentarioFinal":" si su valor es inválido
    (por ejemplo, contiene solo una comilla o está incompleto).
    """
    # Coincide con ,"comentarioFinal":" seguido de cualquier cosa (incluso vacío)
    patron = r',\"comentarioFinal\":\"([^\"]*)\"?'
    match = re.search(patron, texto)

    if match:
        valor = match.group(1)
        # Si el valor está vacío o contiene solo comillas u otros caracteres inválidos
        if valor.strip() == "" or valor.strip() == '"':
            # Lo removemos
            texto = re.sub(patron, '', texto)
    return texto



In [21]:
limpiar_comentario_final('{"usoDelPad":"noPad","comentarioFinal":"}')

'{"usoDelPad":"noPad","comentarioFinal":"}'

In [23]:
serie = df["recorded_at"].dropna().astype(str)
serie = serie[serie.str.strip() != ""]
first_value = serie.iloc[0] if not serie.empty else None

In [24]:
first_value

'2021-09-27 19:00:13'

In [27]:
from datetime import datetime


def get_birth_year(age: str, recorded_at: str) -> int:
    """
    Calcula el año de nacimiento en base a la edad y la fecha registrada.

    :param age: Edad de la persona (str)
    :param recorded_at: Fecha en formato 'YYYY-MM-DD HH:MM:SS'
    :return: Año estimado de nacimiento (int)
    """
    date = datetime.strptime(recorded_at, "%Y-%m-%d %H:%M:%S")
    return date.year - int(age)

In [28]:
get_birth_year(_read_neuropruebas_survey(df)['age'], first_value)

1991